In [1]:
PATH_WORK_DIR = ".."

In [2]:
import warnings
warnings.filterwarnings("ignore")

In [3]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
os.chdir(PATH_WORK_DIR)
print(f"DIRECTORY: {os.getcwd()}")

DIRECTORY: c:\Users\jayar\Desktop\바탕 화면\REPO\PROJECT\M2-PJT_TXT


# packages

In [4]:
import pickle
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
from gensim.corpora import Dictionary
from gensim.models.coherencemodel import CoherenceModel

# data

In [5]:
PATH = './data/national_assembly.pkl'
df = pd.read_pickle(PATH)

In [6]:
DATE_COL = "date"
df[DATE_COL] = pd.to_datetime(df[DATE_COL])
df = (
    df
    .set_index(DATE_COL)
    .sort_index(ascending=True)
)

In [7]:
df.head()

,words
date,
1948-05-31,"[지금, 국회, 회의, 시작, 애국가, 봉창, 국기, 경례, 순국선열, 묵념, 지금..."
1948-06-01,"[국회, 회의, 시작, 국기, 경례, 순국선열, 묵념, 회의, 개회, 자리, 정돈,..."
1948-06-02,"[개회, 착석, 국기, 경례, 묵념, 회의, 개회, 출석, 의원, 이제, 회의록, ..."
1948-06-03,"[개회, 준비, 회의, 개회, 회의록, 낭독, 서기, 회의록, 낭독, 회의록, 이의..."
1948-06-08,"[국회, 개회, 회의록, 통과, 회의록, 낭독, 회의, 교정, 말씀, 교정, 접수,..."


# functions

In [ ]:
def build_coherence_model(
    texts: list, 
    vectorizer: CountVectorizer, 
    lda: LatentDirichletAllocation, 
    top_k: int,
) -> CoherenceModel:
    # gensim dictionary
    dictionary = Dictionary(texts)

    # extract words @ vectorizer
    feature_names = vectorizer.get_feature_names_out()

    # extract top N words @ each topics
    topics = []
    for topic in lda.components_:
        top_idx = topic.argsort()[-top_k:][::-1]
        top_words = [feature_names[i] for i in top_idx]
        topics.append(top_words)

    # coherence model definition
    kwargs = dict(
        topics=topics,
        texts=texts,
        dictionary=dictionary,
        coherence="c_v",
    )
    coherence_model = CoherenceModel(**kwargs)

    return coherence_model

In [ ]:
def engine(
    texts: list, 
    doc_term_mat: csr_matrix, 
    vectorizer: CountVectorizer, 
    date: pd.DatetimeIndex, 
    num_topics: int, 
    top_k: int, 
    seed: int,
) -> tuple[list, dict]:
    # LDA ==========
    kwargs = dict(
        n_components=num_topics,
        random_state=seed,
    )
    lda = LatentDirichletAllocation(**kwargs)

    # DOC-TOPIC MATRIX ==========
    doc_topic_mat = pd.DataFrame(
        data=lda.fit_transform(doc_term_mat),
        index=date,
        columns=[f"k{i+1}" for i in range(num_topics)],
    )

    # COHERENCE ==========
    kwargs = dict(
        texts=texts,
        vectorizer=vectorizer,
        lda=lda,
        top_k=top_k,
    )
    coh = build_coherence_model(**kwargs)
    score = coh.get_coherence()
    score_per_topic = coh.get_coherence_per_topic()

    result = dict(
        lda=lda,
        doc_topic_mat=doc_topic_mat,
        topic_term_mat=lda.components_,
        score_per_topic=score_per_topic,
    )

    return score, result

# modeling

In [10]:
TOKEN_COL = "words"
DATE = df.index
TEXTS = df[TOKEN_COL].tolist()
MIN_DF = 0.01
MAX_DF = 0.5
TOP_K = 10
SEED = 42

In [11]:
kwargs = dict(
    min_df=MIN_DF,
    max_df=MAX_DF,
)
vectorizer = CountVectorizer(**kwargs)

In [12]:
docs = (
    df[TOKEN_COL]
    .apply(lambda x: " ".join(x))
    .tolist()
)
doc_term_mat = vectorizer.fit_transform(docs)

In [ ]:
scores = dict()

for k in range(10,31):
    kwargs = dict(
        texts=TEXTS, 
        doc_term_mat=doc_term_mat, 
        vectorizer=vectorizer, 
        date=DATE,
        num_topics=k, 
        top_k=TOP_K, 
        seed=SEED,
    )
    score, result = engine(**kwargs)

    scores[k] = score

    PATH = f"./result/topic/topic/k{k}.pkl"
    with open(file=PATH, mode="wb") as f:
        pickle.dump(obj=result, file=f)

    print(
        f"NUM TOPIC: {k}",
        f"COH SCORE: {score:.4f}",
        sep="\t",
    )

print("LDA FINISHED")

NUM TOPIC: 10	COH SCORE: 0.6540
NUM TOPIC: 11	COH SCORE: 0.6459
NUM TOPIC: 12	COH SCORE: 0.6179
NUM TOPIC: 13	COH SCORE: 0.6825
NUM TOPIC: 14	COH SCORE: 0.6630
NUM TOPIC: 15	COH SCORE: 0.6523
NUM TOPIC: 16	COH SCORE: 0.6840
NUM TOPIC: 17	COH SCORE: 0.6854
NUM TOPIC: 18	COH SCORE: 0.6705
NUM TOPIC: 19	COH SCORE: 0.6959
NUM TOPIC: 20	COH SCORE: 0.7098
NUM TOPIC: 21	COH SCORE: 0.7020
NUM TOPIC: 22	COH SCORE: 0.6943
NUM TOPIC: 23	COH SCORE: 0.6575
NUM TOPIC: 24	COH SCORE: 0.6781
NUM TOPIC: 25	COH SCORE: 0.7007
NUM TOPIC: 26	COH SCORE: 0.6984
NUM TOPIC: 27	COH SCORE: 0.7026
NUM TOPIC: 28	COH SCORE: 0.7032
NUM TOPIC: 29	COH SCORE: 0.7183
NUM TOPIC: 30	COH SCORE: 0.7063
LDA FINISHED


# save

In [ ]:
PATH = "./result/topic/vectorizer.pkl"
with open(file=PATH, mode="wb") as f:
    pickle.dump(obj=vectorizer, file=f)

In [13]:
PATH = "./result/topic/doc_term_mat.pkl"
with open(file=PATH, mode="wb") as f:
    pickle.dump(obj=doc_term_mat, file=f)

In [ ]:
PATH = "./result/topic/scores.pkl"
with open(file=PATH, mode="wb") as f:
    pickle.dump(obj=scores, file=f)